# Урок 05 - Агентский RAG


## Настройка

В этой тетрадке демонстрируется паттерн Agentic RAG (генерация с дополнением извлечением) с использованием Microsoft Agent Framework.

**Требования:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — конечная точка вашего сервиса поиска Azure AI
- `AZURE_SEARCH_API_KEY` — ключ API вашего сервиса поиска Azure AI
- Развертывание Azure OpenAI, настроенное через переменные окружения
- Аутентификация в Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Что такое Agentic RAG?

Традиционный RAG следует фиксированному процессу: сначала извлечение документов, затем генерация ответа. **Agentic RAG** идёт дальше, предоставляя агенту автономию решать, **когда** и **как** извлекать информацию.

С помощью Agentic RAG агент может:
- **Решать**, нужно ли производить извлечение перед ответом на вопрос
- **Выбирать**, к какому источнику данных или инструменту обращаться
- **Оценивать** извлечённые результаты и выполнять повторные запросы, если первая попытка недостаточна
- **Объединять** информацию из нескольких этапов извлечения в связный ответ

Это делает агента более гибким и точным по сравнению со статичным процессом «извлечь — затем сгенерировать».


## Создание инструмента поиска

В Agentic RAG внешние источники данных обернуты как **инструменты**, которые агент может вызывать по требованию. Это позволяет агенту рассматривать извлечение данных как ещё одно действие, которое он может выполнить, а не как обязательный шаг.

Ниже мы определяем базу знаний о путешествиях и предоставляем её в виде инструмента, который агент может вызывать для поиска информации о назначениях.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Создание агента RAG

Теперь мы создаём агента, которому указано **всегда искать информацию перед ответом**. Агент использует инструмент `search_travel_knowledge`, чтобы основывать свои ответы на базе знаний, а не полагаться на собственные обучающие данные.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итеративный поиск — Шаблон «создатель-проверяющий»

Ключевым преимуществом Agentic RAG является **итеративный поиск**. Агент может выполнять несколько этапов поиска для проверки, уточнения или расширения своих первоначальных результатов — подобно рабочему процессу «создатель-проверяющий»:

1. **Шаг создателя**: Агент получает первоначальную информацию и создает черновик ответа.
2. **Шаг проверяющего**: Агент выполняет дополнительные поиски для проверки деталей или заполнения пробелов.

Ниже агенту задают вопрос, требующий сравнения нескольких направлений, что побуждает его выполнять поиск несколько раз.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Резюме

В этом уроке вы узнали, как создать систему **Agentic RAG** с использованием Microsoft Agent Framework:

- **Agentic RAG** позволяет агентам самостоятельно решать, когда выполнять поиск информации, делая процесс извлечения динамичным, а не фиксированным.
- **Инструменты в качестве источников данных**: Внешние базы знаний (например, Azure AI Search) оборачиваются как инструменты, которые агент может вызывать.
- **Итеративный поиск**: Шаблон maker-checker позволяет агенту выполнять несколько раундов поиска — искать, проверять и уточнять — перед тем, как выдать окончательный ответ.

В производстве вы замените встроенный `TRAVEL_KNOWLEDGE_BASE` на настоящий индекс Azure AI Search для обработки масштабного поиска по туристическим документам.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью AI-сервиса перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для критически важной информации рекомендуется воспользоваться услугами профессионального перевода человеком. Мы не несем ответственности за любые недоразумения или искажения смысла, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
